# Project 2 — Transaction Intelligence Pipeline
## Layer 1: Data Audit
**Purpose:** Understand raw data quality before cleaning  
**Data:** transactions_raw.csv — 1260 rows, 19 columns  
**Date:** 2026-02-27

In [1]:
from datetime import datetime
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import StructField, StructType, StringType

In [2]:
spark = SparkSession.builder \
        .appName("SparkProject2") \
        .master("local[*]") \
        .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/05 10:47:42 WARN Utils: Your hostname, Ashu, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/05 10:47:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 10:47:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Step 1 — Basic Shape, Schema, & Missing Value Analysis

In [3]:
raw_schema = StructType([
    StructField("transaction_id",                   StringType(), True),
    StructField("user_id",                          StringType(), True),
    StructField("account_number",                   StringType(), True),
    StructField("merchant_id",                      StringType(), True),
    StructField("merchant_category",                StringType(), True),
    StructField("transaction_amount",               StringType(), True),
    StructField("currency",                         StringType(), True),
    StructField("transaction_date",                 StringType(), True),
    StructField("payment_channel",                  StringType(), True),
    StructField("transaction_status",               StringType(), True),
    StructField("failure_reason",                   StringType(), True),
    StructField("device_type",                      StringType(), True),
    StructField("location_city",                    StringType(), True),
    StructField("location_country",                 StringType(), True),
    StructField("is_international",                 StringType(), True),
    StructField("bank_branch_code",                 StringType(), True),
    StructField("relationship_manager",             StringType(), True),
    StructField("approval_status",                  StringType(), True),
    StructField("processing_time_ms",               StringType(), True),
])

raw_df = spark.read \
         .option("header", "true") \
         .schema(raw_schema) \
         .csv("/home/ashu/spark-etl-projects/project2-transaction-intelligence/data/transactions_raw.csv")

print(raw_df.head(5))
total_row_count = raw_df.count()
print(f"Total count: {total_row_count}")

expected_nulls = ["failure_reason"]

print("====Null values per column====")
for column in raw_df.columns:
    missing_count = raw_df.filter(
        sf.col(column).isNull() | (sf.col(column) == " ")
    ).count()

    missing_pct = round(missing_count / total_row_count * 100, 2)
    flag = " ← EXPECTED NULL" if column in expected_nulls and missing_count > 0 \
       else " ← PROBLEM" if missing_count > 0 \
       else ""
    print(f"  {column:25s}: {missing_count:4d} missing ({missing_pct}%){flag}")

[Row(transaction_id='TXNSHZXN8BD', user_id='USROQ89V2', account_number='667718320437', merchant_id='MRCUEE4LJ', merchant_category='HEALTHCARE', transaction_amount='43366.98', currency='INR', transaction_date='2026-02-05 15:08:15', payment_channel='UPI', transaction_status='Success', failure_reason=None, device_type='MOBILE', location_city='Bangalore', location_country='India', is_international='False', bank_branch_code='BR0035', relationship_manager='RM020', approval_status='PENDING_REVIEW', processing_time_ms='1818'), Row(transaction_id='TXN6WE7GJ81', user_id='USRDL87XP', account_number='299665013191', merchant_id='MRC4Z1WMJ', merchant_category='EDUCATION', transaction_amount='8755.77', currency='USD', transaction_date='2026/02/11 22:47', payment_channel='IMPS', transaction_status='SUCCESS', failure_reason=None, device_type='WEB', location_city='Singapore', location_country='UAE', is_international='True', bank_branch_code='BR0043', relationship_manager='RM019', approval_status='DECLIN

## Step 3 — Categorical Column Validation

In [4]:
raw_df.groupBy("transaction_status") \
         .agg(sf.count("transaction_status")) \
         .orderBy(sf.col("transaction_status")).show(30, truncate=20) # Has 14 values - Problem

raw_df.groupBy("merchant_category") \
      .agg(sf.count(sf.col("merchant_category"))) \
      .orderBy(sf.col("merchant_category")).show(20, truncate=True) # Has 14 values - Problem

raw_df.groupBy("payment_channel") \
      .agg(sf.count(sf.col("payment_channel"))) \
      .orderBy(sf.col("payment_channel")).show(20, truncate=True) # Has 5 values only

raw_df.groupBy("device_type") \
      .agg(sf.count(sf.col("device_type"))) \
      .orderBy(sf.col("device_type")).show(20, truncate=20) # Has 4 values only

raw_df.groupBy("currency") \
      .agg(sf.count(sf.col("currency"))) \
      .orderBy(sf.col("currency")).show(20, truncate=20) # Has INR + 5 international

raw_df.groupBy("location_country") \
      .agg(sf.count(sf.col("location_country"))) \
      .orderBy(sf.col("location_country")).show(20, truncate=20) # Has India + 5 international

raw_df.groupBy("is_international") \
      .agg(sf.count(sf.col("is_international"))) \
      .orderBy(sf.col("is_international")).show(20, truncate=True) # Has only two values

raw_df.groupBy("approval_status") \
      .agg(sf.count(sf.col("approval_status"))) \
      .orderBy(sf.col("approval_status")).show(20, truncate=20) # Has only 3 values

+------------------+-------------------------+
|transaction_status|count(transaction_status)|
+------------------+-------------------------+
|            FAILED|                       58|
|            Failed|                       74|
|           PENDING|                       38|
|           Pending|                       53|
|          REVERSED|                       18|
|          Reversed|                       15|
|           SUCCESS|                      306|
|           Success|                      295|
|            failed|                       48|
|           pending|                       38|
|          reversed|                       22|
|           success|                      295|
+------------------+-------------------------+

+-----------------+------------------------+
|merchant_category|count(merchant_category)|
+-----------------+------------------------+
|         CLOTHING|                     116|
|        EDUCATION|                     138|
|      ELECTRONICS|   

## Step 4 — Numeric Sanity Check

In [4]:
clean_transaction_amount = sf.regexp_replace(sf.col("transaction_amount"), ",", ".").cast("double")
raw_df.select(
        sf.min(clean_transaction_amount).alias("min_transaction_amount"),
        sf.max(clean_transaction_amount).alias("max_transaction_amount"),
        sf.round(sf.avg(clean_transaction_amount), 2).alias("avg_transaction_amount"),
        sf.sum(sf.when(clean_transaction_amount < 0, 1).otherwise(0)).alias("negative_count")
        ).show()

raw_df.select(
    sf.min(sf.col("processing_time_ms")).alias("min_processing_time"),
    sf.max(sf.col("processing_time_ms")).alias("max_processing_time"),
    sf.sum(sf.when(sf.col("processing_time_ms") < 0, 1).otherwise(0)).alias("negatinve_processing_time_count")
).show()

+----------------------+----------------------+----------------------+--------------+
|min_transaction_amount|max_transaction_amount|avg_transaction_amount|negative_count|
+----------------------+----------------------+----------------------+--------------+
|                100.29|             496125.65|               46231.0|             0|
+----------------------+----------------------+----------------------+--------------+

+-------------------+-------------------+-------------------------------+
|min_processing_time|max_processing_time|negatinve_processing_time_count|
+-------------------+-------------------+-------------------------------+
|               -124|                998|                             30|
+-------------------+-------------------+-------------------------------+



## Step 5 — Date Format Analysis

In [5]:
print("==== TIMESTAMP FORMAT SAMPLES ====")
raw_df.select("transaction_date") \
      .distinct() \
      .limit(10) \
      .show()

today = datetime.now().strftime("%Y-%m-%d")

# This is tricky on raw string dates
# Simple approach — check for year/month patterns
future_date_count = raw_df.filter(
    sf.col("transaction_date").contains("2026-03") |
    sf.col("transaction_date").contains("2026-04") |
    sf.col("transaction_date").contains("2026-05")
).count()

print(f"Future date counts: {future_date_count}")

==== TIMESTAMP FORMAT SAMPLES ====
+-------------------+
|   transaction_date|
+-------------------+
|2026-02-08 04:56:35|
|   2026/01/11 15:59|
|02/12/2026 04:14:06|
|01/27/2026 14:45:38|
|   2026/02/16 06:59|
|   02-02-2026 06:09|
|2026-04-03 09:04:06|
|   14-02-2026 16:33|
|   22-02-2026 20:36|
|01/25/2026 13:11:43|
+-------------------+

Future date counts: 8


## Step 6 — Duplicate Detection

In [6]:
unique_count = raw_df.distinct().count()
print(f"Unique rows: {unique_count}")
duplicate_count = total_row_count - unique_count
print(f"Duplicate rows: {duplicate_count}")

Unique rows: 1200
Duplicate rows: 60


## Step 7 — Business Logic Validation

In [7]:
failed_count = raw_df.filter(
    (sf.upper(sf.col("transaction_status")) == "FAILED") & (sf.col("failure_reason") == "")
).count()

success_count = raw_df.filter(
    (sf.upper(sf.col("transaction_status")) == "SUCCESS") & (sf.col("failure_reason") != "")
).count()

international_currency_count = raw_df.filter(
    (sf.col("is_international") == "True") & (sf.col("currency") == "INR")
).count()

domestic_foreign_currency_count = raw_df.filter(
    (sf.col("is_international") == "False") & (sf.col("currency").isin("AED", "EUR", "GBP", "SGD", "USD"))
).count()

international_country_count = raw_df.filter(
    (sf.col("is_international") == "False") & (sf.col("location_country") != "India")
).count()

print(f"Failed transactions with empty failure reason: {failed_count}")
print(f"Successful transactions with non-empty failure reason: {success_count}")
print(f"International transactions with non-international currency: {international_currency_count}")
print(f"Domestic transactions with non-India country: {domestic_foreign_currency_count}")
print(f"is_international check with non-India country: {international_country_count}")

Failed transactions with empty failure reason: 0
Successful transactions with non-empty failure reason: 0
International transactions with non-international currency: 0
Domestic transactions with non-India country: 0
is_international check with non-India country: 0


## Audit Summary

In [8]:
missing_amount = raw_df.filter(sf.col("transaction_amount").isNull() | (sf.col("transaction_amount") == " ")).count()
missing_merchant_id = raw_df.filter(sf.col("merchant_id").isNull() | (sf.col("merchant_id") == " ")).count()
negative_process_time = raw_df.filter(sf.col("processing_time_ms") < 0).count()
invalid_merchant_category = raw_df.filter(sf.col("merchant_category").isin("N/A", "PETROL BUNK", "general store", "online")).count()
status_variants = raw_df.select("transaction_status").distinct().count()

print("=" * 55)
print("AUDIT SUMMARY")
print("=" * 55)
print(f"Total rows          : {total_row_count}")
print(f"Duplicates          : {duplicate_count}")
print(f"Missing amount      : {missing_amount}")
print(f"Missing merchant_id : {missing_merchant_id}")
print(f"Negative proc time  : {negative_process_time}")
print(f"Invalid merchant cat: {invalid_merchant_category}")
print(f"Status variants     : {status_variants}")
print(f"Future dates        : {future_date_count}")
print(f"Business violations : 0")
print("=" * 55)

AUDIT SUMMARY
Total rows          : 1260
Duplicates          : 60
Missing amount      : 46
Missing merchant_id : 20
Negative proc time  : 30
Invalid merchant cat: 39
Status variants     : 12
Future dates        : 8
Business violations : 0
